### 1) Import

In [57]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

### 2) Cargar Dataset

In [58]:
df = pd.read_csv("../data/adaptive_comfort_dataset.csv")

print("Total filas:", len(df))
df.head()

Total filas: 300


,building_id,neutral_temp,records,cooling_type,region,t_out_mean
0,1,22.585738,170,mixed mode,oceania,15.296857
1,2,22.058339,83,air conditioned,oceania,13.995833
2,3,23.142187,85,air conditioned,americas,0.583480
3,4,23.642083,137,mixed mode,oceania,19.284220
4,5,22.071788,128,air conditioned,americas,9.048210


### 3) Preparar features

In [ ]:
# One-hot encoding para cooling_type
encoder = OneHotEncoder(sparse_output=False)

X_cat = encoder.fit_transform(df[["cooling_type"]])
X_num = df[["t_out_mean"]].values

# Unir variables
X = np.hstack([X_num, X_cat]).astype(np.float32)
y = df["neutral_temp"].values.astype(np.float32)

# pesos (clave)
weights = df["records"].values

### 4) Train/Test split (incluyendo pesos)

In [60]:
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights, test_size=0.2, random_state=42
)

### 5) Modelo Gradient Boosting

In [61]:
model = GradientBoostingRegressor(
    n_estimators=30, learning_rate=0.05, max_depth=3, random_state=42
)

### 6) Entrenamiento con pesos

In [62]:
model.fit(X_train, y_train, sample_weight=w_train)

GradientBoostingRegressor(learning_rate=0.05, n_estimators=30, random_state=42)

### 7) Predicción

In [63]:
y_pred = model.predict(X_test)

### 8) Evaluación sin pesos

In [64]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== SIN PESOS ===")
print("RMSE:", rmse)
print("R2:", r2)

=== SIN PESOS ===
RMSE: 1.8639756427785492
R2: 0.4550473741176406


### 9) Evaluación con pesos

In [65]:
rmse_w = np.sqrt(mean_squared_error(y_test, y_pred, sample_weight=w_test))
r2_w = r2_score(y_test, y_pred, sample_weight=w_test)

print("\n=== CON PESOS ===")
print("RMSE (weighted):", rmse_w)
print("R2 (weighted):", r2_w)


=== CON PESOS ===
RMSE (weighted): 1.5179373345674945
R2 (weighted): 0.5600310560059483


### 10) Prueba con nuevos datos

In [66]:
# ejemplo nuevo
t_out_nuevo = 28.5
cooling_nuevo = "air conditioned"

# preparar input
X_num_new = np.array([[t_out_nuevo]])
X_cat_new = encoder.transform([[cooling_nuevo]])

X_final = np.hstack([X_num_new, X_cat_new]).astype(np.float32)

pred = model.predict(X_final)

print("\nTemperatura neutral predicha:", pred[0])


Temperatura neutral predicha: 24.45335490374234


/home/jorge/anaconda3/envs/sima-ai/lib/python3.10/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
